In [16]:
from sdmetrics.timeseries.utils.distance import jsd, emd

# OpenXC
def openxc_distributional_metrics(raw_df, syn_df):
    res = []
    for col in ['brake_pedal_status', 'transmission_gear_position']:
        res.append(jsd(raw_df[col].to_numpy().reshape(-1, 1), syn_df[col].to_numpy().reshape(-1, 1), categorical_mapping=True))
    for col in ['vehicle_speed', 'engine_speed', 'accelerator_pedal_position']:
        res.append(emd(raw_df[col].to_numpy().reshape(-1, 1), syn_df[col].to_numpy().reshape(-1, 1)))

    return res

In [8]:
import pandas as pd

raw_df = pd.read_csv("../data_selected/openxc/nyc_downtown_east.csv")

syn_df = pd.read_csv("../results/vehiclesec2024/small-scale/csv/realtabformer-tabular_openxc-nyc-downtown-east_20231130183157205073649.csv")

In [3]:
raw_df.shape

(319343, 6)

In [4]:
raw_df.columns

Index(['timestamp', 'brake_pedal_status', 'accelerator_pedal_position',
       'transmission_gear_position', 'vehicle_speed', 'engine_speed'],
      dtype='object')

In [10]:
raw_df['brake_pedal_status'].to_numpy().shape

(319343,)

In [6]:
len(raw_df['brake_pedal_status'].shape)

1

In [17]:
print(openxc_distributional_metrics(raw_df, syn_df))

[0.451917092945644, 0.0018728182427695096, 0.07001700236109762, 4.508375007437142, 0.03024183231196558]


In [24]:
CANGen_BASE_FOLDER = '/ocean/projects/cis230033p/yyin4/CANGen'

DICT_DATASET_FILENAME = {
    # OpenXC
    'openxc-nyc-downtown-east': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'nyc_downtown_east.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'india_New_Delhi_Railway_to_AIIMS.csv'
    ),
    'openxc-taiwan-highwayno2-can': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'taiwan_HighwayNo2_can.csv'
    ),
    'openxc-nyc-downtown-east-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'nyc', 'downtown-east', 'downtown-east_before_imputation.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'india', 'New_Delhi_Railway_to_AIIMS', 'New_Delhi_Railway_to_AIIMS_before_imputation.csv'
    ),
    'openxc-taiwan-highwayno2-can-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'taiwan', 'HighwayNo2-can', 'HighwayNo2-can_before_imputation.csv'
    ),

    # Car-hacking
    'car-hacking-dos-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'DoS_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-fuzzy-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'Fuzzy_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-rpm-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'RPM_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-gear-bits': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'gear_dataset_aligned_train_bits.csv',
    ),
    'car-hacking-dos-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'DoS_dataset_aligned_train.csv',
    ),
    'car-hacking-fuzzy-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'Fuzzy_dataset_aligned_train.csv',
    ),
    'car-hacking-rpm-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'RPM_dataset_aligned_train.csv',
    ),
    'car-hacking-gear-hex': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'car_hacking', 'gear_dataset_aligned_train.csv',
    ),

    # SynCAN
    'syncan-raw': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'syncan', 'train.csv'
    )

}

In [38]:
import os

import numpy as np

for model in ['realtabformer-tabular', 'ctgan']:
    print(model)
    for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
        print(dataset)
        raw_df = pd.read_csv(DICT_DATASET_FILENAME[dataset])
        res = []
        for csv_file in os.listdir("../results/vehiclesec2024/small-scale/csv"):
            if not csv_file.endswith(".csv"):
                continue
            if model == 'realtabformer-tabular' and (not csv_file.startswith(f"{model}_{dataset}_20231130")):
                continue

            if csv_file.startswith(f"{model}_{dataset}_2023"):
                # print(dataset, model, csv_file)

            
                syn_df = pd.read_csv(os.path.join("../results/vehiclesec2024/small-scale/csv", csv_file))

                res.append(openxc_distributional_metrics(raw_df, syn_df))

        assert len(res) == 3 # 3 runs for each configuration

        # print(dataset, model)
        for i in range(5):
            l = []
            for j in range(3):
                l.append(res[j][i])
            # print(f'${np.mean(l):.2f}\pm{np.std(l):.2f}$', end=" & ")
            print(f'${np.mean(l):.2f}\pm{np.std(l):.2f}$')
    # print()

realtabformer-tabular
openxc-nyc-downtown-east
$0.45\pm0.00$
$0.00\pm0.00$
$0.05\pm0.02$
$3.94\pm0.75$
$0.03\pm0.01$
openxc-india-new-delhi-railway-to-aiims
$0.63\pm0.00$
$0.00\pm0.00$
$0.06\pm0.01$
$2.11\pm0.49$
$0.45\pm0.04$
openxc-taiwan-highwayno2-can
$0.68\pm0.00$
$0.00\pm0.00$
$0.06\pm0.04$
$2.17\pm0.24$
$0.05\pm0.00$
ctgan
openxc-nyc-downtown-east
$0.02\pm0.01$
$0.25\pm0.00$
$0.67\pm0.11$
$106.11\pm17.76$
$0.27\pm0.04$
openxc-india-new-delhi-railway-to-aiims
$0.13\pm0.01$
$0.18\pm0.00$
$1.66\pm0.41$
$20.21\pm3.56$
$1.14\pm0.37$
openxc-taiwan-highwayno2-can
$0.22\pm0.01$
$0.28\pm0.00$
$13.18\pm0.30$
$196.89\pm4.94$
$1.82\pm0.17$


In [39]:
for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
    print(dataset)
    raw_df = pd.read_csv(DICT_DATASET_FILENAME[dataset])
    print(set(raw_df['transmission_gear_position']))

openxc-nyc-downtown-east
{'first', 'second', 'fourth', 'neutral', 'third'}
openxc-india-new-delhi-railway-to-aiims
{'fifth', 'first', 'second', 'fourth', 'neutral', 'third'}
openxc-taiwan-highwayno2-can
{'nuetral', 'fifth', 'first', 'second', 'fourth', 'reverse', 'sixth', 'third'}
